# 03a - JSON-Driven Pipeline (mapping.json → Neptune)

Defines the graph schema in JSON, then PipelineExecutor reads it and builds the graph automatically.

No manual transform steps — the JSON defines everything.

In [ ]:
!pip install -q --force-reinstall --no-cache-dir ~/SageMaker/document-graph/wheels/document_graph-3.0.0-py3-none-any.whl

## Define the Schema (JSON)

## Note: Schema JSON can also be loaded from S3

Instead of defining the schema inline, you can load it from S3:

```python
from document_graph.schema.providers.schema_provider_factory import get_schema_provider

provider = get_schema_provider({
    "provider_type": "s3",
    "schema_id": "my-mapping",
    "connection_config": {
        "bucket": "graphrag-artifacts-705909755305",
        "key": "schemas/mapping.json",
        "region": "us-east-1"
    }
})
schema = provider.load_schema()
```

This is useful for sharing schemas across notebooks and pipelines without duplication.


In [ ]:
import json, os

# This is what mapping.json looks like — defines nodes, edges, property mappings
schema = {
    "schema_version": "1.0",
    "graph": {
        "nodes": [
            {"User": {"type": "node", "property": {"name": "user_id", "csv_map": "id"}}},
            {"Account": {"type": "node", "property": {"name": "account_id", "csv_map": "account_id"}}}
        ],
        "edges": [
            {"User": {"BELONGS_TO": "Account"}}
        ]
    },
    "mappings": [
        {"csv_property_name": "id", "type": "node", "arrow_property_name": "user_id", "name": "User"},
        {"csv_property_name": "name", "type": "property", "arrow_property_name": "full_name", "parents": ["User"]},
        {"csv_property_name": "email", "type": "property", "arrow_property_name": "email", "parents": ["User"]},
        {"csv_property_name": "role", "type": "property", "arrow_property_name": "role", "parents": ["User"]},
        {"csv_property_name": "account_id", "type": "node", "arrow_property_name": "account_id", "name": "Account"}
    ]
}

# Save to file
schema_path = os.path.expanduser('~/SageMaker/document-graph/data/users_schema.json')
with open(schema_path, 'w') as f:
    json.dump(schema, f, indent=2)
print(f'Schema written to {schema_path}')
print(f'Nodes: {[list(n.keys())[0] for n in schema["graph"]["nodes"]]}')
print(f'Edges: {schema["graph"]["edges"]}')

## Load Source Data

In [ ]:
import pandas as pd

df = pd.read_csv(os.path.expanduser('~/SageMaker/document-graph/data/users.csv'))
print(f'Source: {len(df)} rows')
print(df)

## Run Pipeline (JSON-driven)

In [ ]:
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from document_graph import PipelineExecutor

GRAPH_STORE = 'neptune-db://obs-app-dev-graph.cluster-czupj1ab2tc6.us-east-1.neptune.amazonaws.com:8182'
gs = GraphStoreFactory.for_graph_store(GRAPH_STORE).__enter__()

executor = PipelineExecutor(gs, tenant_id='schema_driven')
result = executor.run_from_schema(schema_path, source_df=df)

print(f'\n=== Pipeline Result ===')
print(f'Records processed: {result.records_processed}')
print(f'Nodes created: {result.nodes_created}')
print(f'Edges created: {result.edges_created}')
print(f'Errors: {len(result.errors)}')
if result.errors:
    for e in result.errors[:3]:
        print(f'  {e}')

## Verify in Neptune

In [ ]:
from document_graph.query import DocumentGraphQueryEngine

engine = DocumentGraphQueryEngine(gs, tenant_id='schema_driven')

print('Users:')
for u in engine.get_nodes('User'):
    props = u['n']['~properties']
    print(f"  {props.get('full_name', props.get('user_id', '?'))} ({props.get('role', '?')})")

print('\nAccounts:')
for a in engine.get_nodes('Account'):
    props = a['n']['~properties']
    print(f"  {props.get('account_id', '?')}")

print(f'\n✅ JSON-driven pipeline complete — schema → Neptune in one call')